# 🧮 GLM-OCR v5.0 — SOTA Handwritten Math → Compilable LaTeX
### Fine-Tuned Vision-Language Model on Held-Out University Math Exam Corpus
[![GitHub](https://img.shields.io/badge/GitHub-realgauravvyas%2Focr2tex-181717?logo=github)](https://github.com/realgauravvyas/ocr2tex)
[![HuggingFace](https://img.shields.io/badge/HuggingFace-realgauravvyas-FFD21E?logo=huggingface)](https://huggingface.co/realgauravvyas)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)
**GLM-OCR v5.0** is an advanced multimodal OCR model fine-tuned on **12,575+ undergraduate handwritten mathematics answer sheets** using [zai-org/GLM-OCR](https://huggingface.co/zai-org/GLM-OCR) and LoRA adapters.
### 🏆 Benchmark Highlights on Held-Out Test Set (250 pages evaluated):
- **Mean CER:** `0.3377` *(~34.4% error reduction vs base model!)*
- **Normalized CER:** `0.3683` *(All-time project best)*
- **BLEU-4:** `0.6594` | **Math-F1:** `0.8358` | **chrF:** `0.7539`
- **Clean PDF Compile Rate:** **82.0%** *(vs 0.0% on base GLM-OCR)*


---
## Step 1 — Check GPU Acceleration
Colab provides a free **T4 GPU** (or V100/A100 with Colab Pro). Ensure CUDA is active before proceeding.
*If no GPU is found: Go to **Runtime → Change runtime type → T4 GPU → Save** and re-run this cell.*



In [ ]:
import torch

assert torch.cuda.is_available(), (
    "❌ No GPU detected! Please change runtime: Runtime -> Change runtime type -> T4 GPU -> Save"
)
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU Active: {gpu_name} ({vram_gb:.2f} GB VRAM)")



---
## Step 2 — Install Dependencies & LaTeX Engine (~3-4 min)
Installs:
1. Multimodal PyTorch libraries (`transformers`, `peft`, `accelerate`, `torchao`, `pillow`, `pymupdf`).
2. Minimal TeX Live environment (`pdflatex`) to compile and render predicted LaTeX directly to PDF.



In [ ]:
%%time
import sys

print("📦 Installing Python packages...")
!pip install -q -U transformers peft accelerate torchao>=0.16.0 "pillow<11.0.0" pymupdf gdown

print("📄 Installing TeX Live compiler (for PDF rendering)...")
!apt-get update -qq && apt-get install -y -qq texlive-latex-base texlive-latex-extra texlive-fonts-recommended > /dev/null 2>&1

# Verify pdflatex
!pdflatex -version | head -1
print("✅ Environment ready!")



---
## Step 3 — Download GLM-OCR v5.0 Package from Google Drive
Paste your **Google Drive Shareable File ID** below.
The package contains:
- `adapter/`: LoRA adapter weights (`adapter_model.safetensors`, tokenizer configs)
- `test_manifest.json`: Reference metadata & ground-truth LaTeX for all 700 held-out test pages
- `test_images/`: Held-out handwritten test images (pages 1 to 700)



In [ ]:
import os, zipfile, shutil
from pathlib import Path
import gdown

#@title 📥 Configure Google Drive File ID
# Paste your Google Drive File ID below (from the shareable link: drive.google.com/file/d/<FILE_ID>/view)
ADAPTER_DRIVE_FILE_ID = "YOUR_GOOGLE_DRIVE_FILE_ID_HERE" #@param {type:"string"}

PACKAGE_ZIP = Path("/content/glm_ocr_v5_package.zip")
EXTRACT_DIR = Path("/content/glm_ocr_v5")
ADAPTER_DIR = EXTRACT_DIR / "adapter"
TEST_IMAGES_DIR = EXTRACT_DIR / "test_images"
MANIFEST_FILE = EXTRACT_DIR / "test_manifest.json"

if not ADAPTER_DIR.exists():
    if ADAPTER_DRIVE_FILE_ID and "YOUR_" not in ADAPTER_DRIVE_FILE_ID:
        print(f"📥 Downloading package from Google Drive (ID: {ADAPTER_DRIVE_FILE_ID})...")
        url = f"https://drive.google.com/uc?id={ADAPTER_DRIVE_FILE_ID}"
        gdown.download(url, str(PACKAGE_ZIP), quiet=False)
    else:
        print("⚠️ No valid Google Drive ID provided. You can upload 'glm_ocr_v5_package.zip' manually via the files tab.")

    if PACKAGE_ZIP.exists():
        print("📂 Extracting package...")
        EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(PACKAGE_ZIP, "r") as z:
            z.extractall(EXTRACT_DIR)
        print("✅ Extraction complete!")
    else:
        print("❌ Waiting for package zip upload. Once uploaded to /content/glm_ocr_v5_package.zip, re-run this cell.")
else:
    print(f"✅ Adapter already extracted at: {ADAPTER_DIR}")



---
## Step 4 — Load GLM-OCR Base Model & Attach v5.0 LoRA Adapter (~2 min)
We load `zai-org/GLM-OCR` in `bfloat16` and attach the fine-tuned LoRA adapter.
This architecture allows comparing **Base GLM-OCR (zero-shot)** against **v5.0 Fine-Tuned** in the same session without extra memory!



In [ ]:
%%time
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel

MODEL_NAME = "zai-org/GLM-OCR"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading Base GLM-OCR ({MODEL_NAME}) in bfloat16...")
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
).to(device)

print(f"Attaching Fine-Tuned LoRA Adapter: {ADAPTER_DIR}...")
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR)).to(device)
model.eval()
print("✅ Model & v5.0 Adapter successfully loaded into GPU VRAM!")



---
## Step 5 — Helper Functions: Generation, Compilation & Side-by-Side Visualizer
Functions:
- `generate_latex(image, use_adapter=True)`: Runs multimodal autoregressive decoding.
- `compile_latex_to_pdf(tex_code, output_stem)`: Uses `pdflatex` to build a clean PDF and renders high-res page image via `pymupdf`.
- `levenshtein_cer(ref, hyp)`: Live Character Error Rate computation.



In [ ]:
import io, re, subprocess, tempfile, html
from PIL import Image
import fitz # PyMuPDF
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

USER_PROMPT = (
    "OCR this handwritten math page. Convert ONLY the handwritten mathematical "
    "content into a complete, compilable LaTeX document. Ignore printed text, "
    "student info, page numbers, cancelled work and rough work. Output only LaTeX."
)

RESULTS_DIR = Path("/content/results")
RESULTS_DIR.mkdir(exist_ok=True)

def levenshtein_cer(ref: str, hyp: str) -> float:
    """Calculates Levenshtein Character Error Rate (CER)."""
    ref, hyp = " ".join(ref.split()), " ".join(hyp.split())
    if not ref: return 0.0 if not hyp else 1.0
    r_arr = np.frombuffer(ref.encode("utf-32-le"), dtype=np.uint32)
    h_arr = np.frombuffer(hyp.encode("utf-32-le"), dtype=np.uint32)
    n = len(h_arr)
    prev = np.arange(n + 1, dtype=np.int64)
    for i in range(len(r_arr)):
        cand = np.empty(n + 1, dtype=np.int64)
        cand[0] = i + 1
        cand[1:] = np.minimum(prev[1:] + 1, prev[:-1] + (h_arr != r_arr[i]))
        running = np.minimum.accumulate(cand - np.arange(n + 1))
        prev = np.minimum(cand, running + np.arange(n + 1))
    return round(float(prev[-1]) / len(r_arr), 4)

def generate_latex(img: Image.Image, use_adapter=True, max_new_tokens=1024) -> str:
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": USER_PROMPT}]}]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt], images=[img], return_tensors="pt").to(device)

    # Enable or disable adapter on the fly
    if hasattr(model, "set_adapter"):
        if use_adapter:
            model.enable_adapters()
        else:
            model.disable_adapters()

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = out[0][inputs["input_ids"].shape[1]:]
    return processor.tokenizer.decode(gen, skip_special_tokens=True).strip()

def compile_latex_to_pdf(tex_str: str, out_prefix: str, timeout=25):
    """Compiles LaTeX string to PDF and renders page 1 as PIL Image."""
    if not tex_str or "\\documentclass" not in tex_str:
        return False, None, "No \\documentclass found"
    with tempfile.TemporaryDirectory() as td:
        tex_path = Path(td) / "doc.tex"
        tex_path.write_text(tex_str, encoding="utf-8")
        try:
            res = subprocess.run(
                ["pdflatex", "-interaction=nonstopmode", "-halt-on-error", "doc.tex"],
                cwd=td, capture_output=True, text=True, timeout=timeout
            )
        except Exception as e:
            return False, None, str(e)

        pdf_path = Path(td) / "doc.pdf"
        if pdf_path.exists():
            # Save permanent PDF
            perm_pdf = RESULTS_DIR / f"{out_prefix}.pdf"
            perm_pdf.write_bytes(pdf_path.read_bytes())
            
            # Render first page as Image
            doc = fitz.open(pdf_path)
            page = doc.load_page(0)
            pix = page.get_pixmap(dpi=150)
            img = Image.open(io.BytesIO(pix.tobytes("png")))
            return True, img, None
        return False, None, res.stdout[-300:] if res.stdout else "Compilation error"



---
## Step 6 — 🎯 Interactive Testing: Test Any Page (1 to 700) or Random
Select:
- **`Test_Mode`**:
  - `Specific Page (1-700)`: Enter an exact page number.
  - `Random Page`: Randomly draws from the 700 held-out test split.
- **`Compare_With_Base`**: Toggle to run Base GLM-OCR alongside v5.0!



In [ ]:
import random, json

#@title 🔍 Page Selector & Tester
Test_Mode = "Specific Page (1-700)" #@param ["Specific Page (1-700)", "Random Page"]
Page_Number = 1 #@param {type:"slider", min:1, max:700, step:1}
Compare_With_Base = True #@param {type:"boolean"}

# Load manifest
manifest = []
if MANIFEST_FILE.exists():
    with open(MANIFEST_FILE, "r", encoding="utf-8") as f:
        manifest = json.load(f)

if not manifest:
    print("⚠️ Manifest not found. Ensure Step 3 package was extracted.")
else:
    if Test_Mode == "Random Page":
        chosen_idx = random.randint(1, len(manifest))
    else:
        chosen_idx = int(Page_Number)

    entry = manifest[chosen_idx - 1]
    sid = entry["id"]
    ref_latex = entry.get("ground_truth", "")
    img_file = TEST_IMAGES_DIR / f"{sid}.png"

    print(f"📄 Testing Page [{chosen_idx}/700] | ID: {sid}")
    if not img_file.exists():
        print(f"❌ Image not found at {img_file}")
    else:
        test_img = Image.open(img_file).convert("RGB")
        
        # 1. Run v5.0
        print("⚡ Running GLM-OCR v5.0 Inference...")
        hyp_v5 = generate_latex(test_img, use_adapter=True)
        v5_ok, v5_pdf_img, v5_err = compile_latex_to_pdf(hyp_v5, f"{sid}_v5")
        cer_v5 = levenshtein_cer(ref_latex, hyp_v5) if ref_latex else None

        # 2. Run Base Model (Optional)
        hyp_base, base_ok, base_pdf_img = None, False, None
        cer_base = None
        if Compare_With_Base:
            print("⚪ Running Base GLM-OCR Inference...")
            hyp_base = generate_latex(test_img, use_adapter=False)
            base_ok, base_pdf_img, base_err = compile_latex_to_pdf(hyp_base, f"{sid}_base")
            cer_base = levenshtein_cer(ref_latex, hyp_base) if ref_latex else None

        # 3. Visual Display
        cols = 3 if Compare_With_Base else 2
        fig, axes = plt.subplots(1, cols, figsize=(7 * cols, 10))
        if cols == 2: axes = [axes[0], axes[1]]

        # Col 1: Original Image
        axes[0].imshow(test_img)
        axes[0].set_title(f"Input Handwritten: {sid}", fontsize=14, fontweight="bold")
        axes[0].axis("off")

        # Col 2: v5.0 Output
        if v5_ok and v5_pdf_img:
            axes[1].imshow(v5_pdf_img)
            axes[1].set_title(f"GLM-OCR v5.0 (PDF Compiled ✅)
CER: {cer_v5}", fontsize=14, fontweight="bold", color="green")
        else:
            axes[1].text(0.5, 0.5, f"Compilation Failed ❌\n{v5_err[:100]}", ha="center", va="center", color="red")
            axes[1].set_title(f"GLM-OCR v5.0 (Compile Failed ❌)\nCER: {cer_v5}", fontsize=14, fontweight="bold")
        axes[1].axis("off")

        # Col 3: Base Output
        if Compare_With_Base:
            if base_ok and base_pdf_img:
                axes[2].imshow(base_pdf_img)
                axes[2].set_title(f"Base GLM-OCR (PDF Compiled ✅)\nCER: {cer_base}", fontsize=14, fontweight="bold")
            else:
                axes[2].text(0.5, 0.5, "Base Output Failed to Compile ❌", ha="center", va="center", color="red")
                axes[2].set_title(f"Base GLM-OCR (Compile Failed ❌)\nCER: {cer_base}", fontsize=14, fontweight="bold")
            axes[2].axis("off")

        plt.tight_layout()
        plt.show()

        # Print Copyable LaTeX Box
        v5_safe = html.escape(hyp_v5)
        display(HTML(f"""
        <div style="border: 1px solid #00c853; border-radius: 8px; padding: 15px; margin-top: 15px; background-color: #f9fbf9;">
            <h4 style="color: #2e7d32; margin-top: 0;">📋 GLM-OCR v5.0 Output LaTeX (Copyable)</h4>
            <textarea style="width: 100%; height: 180px; font-family: monospace; font-size: 13px; padding: 8px;" readonly>{v5_safe}</textarea>
        </div>
        """))



---
## Step 7 — 📤 Upload Your Own Custom Math Page
Upload any photo or scan of your handwritten math notes (`.png`, `.jpg`) to convert it to LaTeX and compile to PDF!



In [ ]:
from google.colab import files

UPLOAD_DIR = Path("/content/my_uploads")
UPLOAD_DIR.mkdir(exist_ok=True)

print("Select one or more handwritten math images...")
uploaded = files.upload()

for filename, data in uploaded.items():
    file_path = UPLOAD_DIR / filename
    file_path.write_bytes(data)
    custom_img = Image.open(file_path).convert("RGB")
    
    print(f"
Processing Custom Upload: {filename}...")
    latex_out = generate_latex(custom_img, use_adapter=True)
    comp_ok, pdf_img, err_msg = compile_latex_to_pdf(latex_out, f"custom_{file_path.stem}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 10))
    axes[0].imshow(custom_img)
    axes[0].set_title("Uploaded Handwritten Page", fontsize=14, fontweight="bold")
    axes[0].axis("off")

    if comp_ok and pdf_img:
        axes[1].imshow(pdf_img)
        axes[1].set_title("GLM-OCR v5.0 Compiled PDF Result ✅", fontsize=14, fontweight="bold", color="green")
    else:
        axes[1].text(0.5, 0.5, f"Compilation Failed ❌\n{err_msg[:120]}", ha="center", va="center", color="red")
        axes[1].set_title("Compilation Error", fontsize=14, fontweight="bold")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

    latex_escaped = html.escape(latex_out)
    display(HTML(f"""
    <div style="border: 1px solid #1565c0; border-radius: 8px; padding: 15px; margin-top: 15px; background-color: #f5f8fc;">
        <h4 style="color: #0d47a1; margin-top: 0;">📋 Generated LaTeX for {filename}</h4>
        <textarea style="width: 100%; height: 180px; font-family: monospace; font-size: 13px; padding: 8px;" readonly>{latex_escaped}</textarea>
    </div>
    """))



---
## Step 8 — 💾 Download All Results
Bundles all generated `.tex` and `.pdf` files from this session into a single `.zip` file.



In [ ]:
import shutil
from google.colab import files as colab_files

results_count = len(list(RESULTS_DIR.glob("*")))
if results_count > 0:
    zip_path = shutil.make_archive("/content/glm_ocr_v5_results", "zip", RESULTS_DIR)
    print(f"📦 Zipped {results_count} result files!")
    colab_files.download(zip_path)
else:
    print("No results generated yet. Run Step 6 or Step 7 first.")



---
## 📚 Citation & Author Information
- **Author:** Gaurav Vyas
- **Hugging Face:** [@realgauravvyas](https://huggingface.co/realgauravvyas)
- **GitHub Repository:** [realgauravvyas/ocr2tex](https://github.com/realgauravvyas/ocr2tex)
- **Base Architecture:** [zai-org/GLM-OCR (0.9B)](https://huggingface.co/zai-org/GLM-OCR)

